# Lab 11

In this lab we will try some more modifications of the Kalman filter from Lecture 22. The intro text here is the same as from that lecture, but is included for reference.

In the first exampke, we will use a physics-based model to estimate a 2D trajectory, so we already know how $\Phi$ and $A$. 

For example, we know that we can get position directly at time $t$ from $p_t = p_{t-1} + \Delta t v_{t-1}$. Thus, $\Phi$ and $A$ are specified here rather than estimated. In other applications, like the blood markers we talked about last time, we'd have to estimate $\Phi$ from the data.

**Model**

State: $x_t = [p_x, p_y, v_x, v_y]^\top$ (position and velocity)

$$x_t = \Phi x_{t-1} + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$
$$y_t = A x_t + v_t, \quad v_t \sim \mathcal{N}(0, R)$$

with
$$\Phi = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \quad A = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$

This $\Phi$ matrix can be interpreted as showing that position updates by adding $\Delta t v$ to our position, but velocity is unchanged from one step to the next (as evidenced by 1's in the bottom right diagonal). 

We observe noisy position only for $y_t$, while velocity is unobserved (why we have zeros in $A$ for the last two variables). The filter has to infer velocity from the position history.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, Layout, Dropdown
from IPython.display import display

np.set_printoptions(precision=3, suppress=True)

## Ground truth

Let's show what happens when the trajectory is a gentle curve. This matters because the filter's constant-velocity assumption is wrong, and the true velocity is in fact changing over time. This will allow us to show what happens when the model is misspecified.

In [ ]:
def gen_truth(N=60):
    """
    Generate a gentle curve
    """
    s = np.linspace(0, 1, N)
    x = -8 + 16 * s
    y = 2 * np.sin(1.3 * np.pi * s) - 1 * s
    return np.stack([x, y], axis=1)


def gen_measurements(truth, sigma_R, seed):
    """
    Make noisy measurements from our ground truth with some measurement
    error with covariance sigma_R
    """
    rng = np.random.default_rng(seed)
    return truth + sigma_R * rng.standard_normal(truth.shape)

## Kalman filter

Now we will build the state-space matrices and create functions for both the Kalman Filter and the Kalman Smoother. In this part, we will extend the model so that we can build matrices both for the original `model='constant_velocity'` and a new model, `model='constant_acceleration'`. For the constant acceleration model, we will use the following kinematics:

$$
\begin{aligned}
p &= p_0 + v\Delta t + \frac{1}{2}a \Delta t^2\\
v &= v_0 + a\Delta t\\
a &= a_0
\end{aligned}
$$

So you will use these equations to fill in the $\Phi$ and $A$ matrices for the `'constant_acceleration'` case in `build_matrices()`.

In [ ]:
def build_matrices(sigma_Q, sigma_R, dt=1.0, model='constant_velocity'):
    """Build state-space matrices for 2D tracking.

    Two models available:
    - 'constant_velocity': state = [px, py, vx, vy]
      Assumes velocity persists, and process noise models unmodeled acceleration.
    - 'constant_acceleration': state = [px, py, vx, vy, ax, ay]
      Assumes acceleration persists; process noise models unmodeled jerk.
      Better for curved trajectories where the constant-velocity assumption
      is misspecified. 

    Parameters
    ----------
    sigma_Q : float
        Process noise scale. For constant_velocity, this is unmodeled
        acceleration stdev. For constant_acceleration, this is unmodeled
        jerk stdev.
    sigma_R : float
        Measurement noise standard deviation (position, isotropic).
    dt : float, default 1.0
        Time step.
    model : str, default 'constant_velocity'
        Either 'constant_velocity' or 'constant_acceleration'.

    Returns
    -------
    Phi : ndarray, (4,4) or (6,6)
    Q   : ndarray, matching Phi
    A   : ndarray, (2,4) or (2,6)
    R   : ndarray, (2,2)
    """
    if model == 'constant_velocity': # This is what we used in class
        Phi = np.array([[1, 0, dt, 0],
                        [0, 1, 0, dt],
                        [0, 0, 1,  0],
                        [0, 0, 0,  1]], dtype=float)
        q = sigma_Q ** 2
        Q = q * np.array([[dt**3/3, 0,       dt**2/2, 0      ],
                          [0,       dt**3/3, 0,       dt**2/2],
                          [dt**2/2, 0,       dt,      0      ],
                          [0,       dt**2/2, 0,       dt     ]])
        A = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0]], dtype=float)

    elif model == 'constant_acceleration':
        ######## FILL IN ######
        Phi = ## FILL IN
        
        # Continuous white-noise jerk discretization
        # Block structure: each 1D block for [p, v, a] driven by jerk noise
        q = sigma_Q ** 2
        Q_1d = q * np.array([[dt**5/20, dt**4/8, dt**3/6],
                             [dt**4/8,  dt**3/3, dt**2/2],
                             [dt**3/6,  dt**2/2, dt     ]])
        Q = np.zeros((6, 6))
        Q[0::2, 0::2] = Q_1d  # x-components: indices 0, 2, 4
        Q[1::2, 1::2] = Q_1d  # y-components: indices 1, 3, 5

        ######## FILL IN ######
        A = ## FILL IN
        
    else:
        raise ValueError(f"Unknown model: {model}")

    R = sigma_R ** 2 * np.eye(2)
    return Phi, Q, A, R


def kalman_filter(meas, sigma_Q, sigma_R, model='constant_velocity'):
    """Run the Kalman filter forward pass on a sequence of 2D position measurements.

    At each time t, performs:
        Predict:  x_t^{t-1} = Phi @ x_{t-1}^{t-1}
                  P_t^{t-1} = Phi @ P_{t-1}^{t-1} @ Phi.T + Q
        Update:   K_t = P_t^{t-1} @ A.T @ (A @ P_t^{t-1} @ A.T + R)^{-1}
                  x_t^t = x_t^{t-1} + K_t @ (y_t - A @ x_t^{t-1})
                  P_t^t = (I - K_t @ A) @ P_t^{t-1}

    Initialization uses the first measurement as the position estimate. 
    This is a practical shortcut; a mathematically
    cleaner alternative is a diffuse prior (P_0 very large), which produces
    nearly identical results after the first few steps.

    Parameters
    ----------
    meas : (N, 2) ndarray
        Noisy 2D position observations.
    sigma_Q : float
        Process noise scale (see build_matrices).
    sigma_R : float
        Measurement noise standard deviation (see build_matrices).

    Returns
    -------
    xf : (N, D) ndarray
        Filtered state means x_t^t = E[x_t | y_{1:t}].
    Pf : (N, D, D) ndarray
        Filtered state covariances P_t^t.
    xp : (N, D) ndarray
        One-step-ahead predicted state means x_t^{t-1} = E[x_t | y_{1:t-1}].
        At t=0 this equals the initial state (no prediction step).
    Pp : (N, D, D) ndarray
        One-step-ahead predicted state covariances P_t^{t-1}. Required
        inputs for the RTS smoother.
    """
    Phi, Q, A, R = build_matrices(sigma_Q, sigma_R, model=model)
    N = len(meas)
    d = Phi.shape[0] # State dimension - 4 or 6
    
    # Initial state: first measurement for position, zero for velocity (and 
    # possibly acceleration if included in the model)
    x = np.zeros(d)
    x[0] = meas[0,0]
    x[1] = meas[0,1]

    P = np.eye(d)
    P[2:,2:] *= 10 # large initial uncertainty on velocity (and maybe acceleration)
    xf = np.zeros((N, d));  Pf = np.zeros((N, d, d))
    xp = np.zeros((N, d));  Pp = np.zeros((N, d, d))
    for t in range(N):
        if t > 0:
            # Predict
            x = Phi @ x
            P = Phi @ P @ Phi.T + Q
        xp[t] = x;  Pp[t] = P
        
        # Update
        innov = meas[t] - A @ x                # innovation
        S = A @ P @ A.T + R                    # innovation covariance
        K = P @ A.T @ np.linalg.inv(S)         # Kalman gain
        x = x + K @ innov
        P = (np.eye(d) - K @ A) @ P
        xf[t] = x;  Pf[t] = P
    return xf, Pf, xp, Pp


def rts_smoother(xf, Pf, xp, Pp, sigma_Q, model='constant_velocity'):
    """Run the Rauch-Tung-Striebel smoother backward pass.

    Computes x_t^N = E[x_t | y_{1:N}] and P_t^N using the full observation
    record. Initialized at t=N-1 with the filtered estimate (where filter
    and smoother coincide), then iterates backward using:

        J_t = P_t^t @ Phi.T @ (P_{t+1}^t)^{-1}
        x_t^N = x_t^t + J_t @ (x_{t+1}^N - x_{t+1}^t)
        P_t^N = P_t^t + J_t @ (P_{t+1}^N - P_{t+1}^t) @ J_t.T

    The smoother's error covariance P_t^N is guaranteed to satisfy
    tr(P_t^N) <= tr(P_t^t) for all t, since smoothing conditions on strictly
    more information.

    Parameters
    ----------
    xf : (N, D) ndarray
        Filtered state means from kalman_filter.
    Pf : (N, D, D) ndarray
        Filtered state covariances from kalman_filter.
    xp : (N, D) ndarray
        One-step-ahead predicted means from kalman_filter.
    Pp : (N, D, D) ndarray
        One-step-ahead predicted covariances from kalman_filter.
    sigma_Q : float
        Process noise scale. Must match the value used in the forward pass.
        Phi does not actually depend on sigma_Q in this constant-velocity
        model; the argument is kept for interface consistency.
    model : str, default 'constant_velocity'
        Either 'constant_velocity' or 'constant_acceleration'.

    Returns
    -------
    xs : (N, D) ndarray 
        Smoothed state means x_t^N = E[x_t | y_{1:N}]. 
    Ps : (N, D, D) ndarray
        Smoothed state covariances P_t^N.
    """
    Phi, _, _, _ = build_matrices(sigma_Q, 1.0, model=model)  # R doesn't matter here
    N = len(xf)
    xs = xf.copy();  Ps = Pf.copy()
    for t in range(N - 2, -1, -1):
        J = Pf[t] @ Phi.T @ np.linalg.inv(Pp[t+1])
        xs[t] = xf[t] + J @ (xs[t+1] - xp[t+1])
        Ps[t] = Pf[t] + J @ (Ps[t+1] - Pp[t+1]) @ J.T
    return xs, Ps

## Sanity check

Run once with default parameters and plot.

In [ ]:
truth = gen_truth()
meas = gen_measurements(truth, sigma_R=0.8, seed=1)
xf, Pf, xp, Pp = kalman_filter(meas, sigma_Q=0.05, sigma_R=0.8, model='constant_acceleration')
xs, Ps = rts_smoother(xf, Pf, xp, Pp, sigma_Q=0.05, model='constant_acceleration')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(truth[:, 0], truth[:, 1], 'k-', lw=1.5, label='Truth')
ax[0].plot(meas[:, 0], meas[:, 1], '.', color='#D85A30', ms=5, alpha=0.75, label='Measurements')
ax[0].plot(xf[:, 0], xf[:, 1], '-', color='#378ADD', lw=1.8, label='Filter')
ax[0].plot(xs[:, 0], xs[:, 1], '--', color='#1D9E75', lw=1.8, label='Smoother')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y'); ax[0].legend(frameon=False, fontsize=9)
ax[0].set_title('Position')

tr_f = np.trace(Pf[:, :2, :2], axis1=1, axis2=2)
tr_s = np.trace(Ps[:, :2, :2], axis1=1, axis2=2)
ax[1].plot(tr_f, '-', color='#378ADD', lw=1.8, label='Filter')
ax[1].plot(tr_s, '--', color='#1D9E75', lw=1.8, label='Smoother')
ax[1].set_xlabel('t'); ax[1].set_ylabel(r'$\mathrm{tr}(P^{pos}_t)$'); ax[1].legend(frameon=False, fontsize=9)
ax[1].set_title('Position uncertainty')
plt.tight_layout(); plt.show()

## Interactive version

Use the sliders to explore. The covariance ellipses are 95% confidence regions from the 2×2 position block of $P$.

### Questions for you to try answering:

1. **Default values** ($\sigma_Q = 0.05$, $\sigma_R = 0.80$). What do you predict the filter will do?
2. **Large $\sigma_R$** (~2.5): measurements become nearly useless. Does the filter still track? What will the estimate look like?
3. **Tiny $\sigma_Q$** (~0.005): filter is told to trust its dynamics model. What goes wrong, and why?
4. **Enable the smoother.** Where does it help most — beginning, middle, end?
5. **Adding acceleration.** How does adding acceleration terms to $\Phi$ help the model in different scenarios?
6. **Changing the ground truth.** Play around with the `gen_truth` function at the top and observe how this affects your estimates. Which can be estimated with constant_velocity vs. constant_acceleration?

In [ ]:
def cov_ellipse(ax, mean, cov, color, alpha=0.12):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    # chi^2 with 2 dof, 95%: 5.991
    w, h = 2 * np.sqrt(5.991 * vals)
    e = Ellipse(mean, w, h, angle=theta, facecolor=color, alpha=alpha,
                edgecolor=color, lw=0.8)
    ax.add_patch(e)

def demo(sigma_Q=0.05, sigma_R=0.8, seed=1, model='constant_velocity',
         show_measurements=True, show_filter=True,
         show_smoother=False, show_ellipses=True):
    truth = gen_truth()
    meas = gen_measurements(truth, sigma_R, seed)
    
    xf, Pf, xp, Pp = kalman_filter(meas, sigma_Q, sigma_R, model=model)
    xs, Ps = rts_smoother(xf, Pf, xp, Pp, sigma_Q, model=model)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5),
                           gridspec_kw={'width_ratios': [2, 1]})

    # Position plot
    ax[0].plot(truth[:, 0], truth[:, 1], 'k-', lw=1.5, label='Truth')
    ax[0].annotate('t=0', xy=(truth[0, 0], truth[0, 1]),
                   xytext=(8, 8), textcoords='offset points',
                   fontsize=9, color='black')
    if show_measurements:
        ax[0].plot(meas[:, 0], meas[:, 1], '.', color='#D85A30',
                   ms=5, alpha=0.75, label='Measurements')
    if show_filter:
        ax[0].plot(xf[:, 0], xf[:, 1], '-', color='#378ADD',
                   lw=1.8, label='Filter')
    if show_smoother:
        ax[0].plot(xs[:, 0], xs[:, 1], '--', color='#1D9E75',
                   lw=1.8, label='Smoother')
    if show_ellipses:
        stride = 6
        for t in range(0, len(truth), stride):
            if show_filter:
                cov_ellipse(ax[0], xf[t, :2], Pf[t, :2, :2], '#378ADD')
            if show_smoother:
                cov_ellipse(ax[0], xs[t, :2], Ps[t, :2, :2], '#1D9E75')

    #ax[0].set_xlim(-10, 10); ax[0].set_ylim(-4, 4)
    ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
    ax[0].legend(frameon=False, fontsize=9, loc='lower left')
    ax[0].set_title(f'Position   σ_Q={sigma_Q:.3f}, σ_R={sigma_R:.2f}')

    # Trace plot
    # This gives you the uncertainty for 2D collapsed into
    # a single scalar
    tr_f = np.trace(Pf[:, :2, :2], axis1=1, axis2=2)
    tr_s = np.trace(Ps[:, :2, :2], axis1=1, axis2=2)
    if show_filter:
        ax[1].plot(tr_f, '-', color='#378ADD', lw=1.8, label='Filter')
    if show_smoother:
        ax[1].plot(tr_s, '--', color='#1D9E75', lw=1.8, label='Smoother')
    ax[1].set_xlabel('t'); ax[1].set_ylabel(r'$\mathrm{tr}(P^{pos}_t)$')
    ax[1].legend(frameon=False, fontsize=9)
    ax[1].set_title('Position uncertainty')

    plt.tight_layout(); plt.show()

style = {'description_width': '160px'}
layout = Layout(width='500px')
interact(demo,    
         model=Dropdown(options=['constant_velocity', 'constant_acceleration'],
                        value='constant_velocity', description='Model',
                        style=style, layout=layout),
         sigma_Q=FloatSlider(value=0.05, min=0.001, max=0.5, step=0.001,
                                description='Process noise σ_Q', readout_format='.3f',
                                style=style, layout=layout),
         sigma_R=FloatSlider(value=0.8, min=0.05, max=3.0, step=0.01,
                                description='Measurement noise σ_R', readout_format='.2f',
                                style=style, layout=layout),
         seed=IntSlider(value=1, min=1, max=20, step=1, description='seed'),
         show_measurements=Checkbox(value=True, description='Measurements'),
         show_filter=Checkbox(value=True, description='Filter'),
         show_smoother=Checkbox(value=False, description='Smoother'),
         show_ellipses=Checkbox(value=True, description='Ellipses'));

## Questions for you to try answering (repeated):

1. **Default values** ($\sigma_Q = 0.05$, $\sigma_R = 0.80$). What do you predict the filter will do?
2. **Large $\sigma_R$** (~2.5): measurements become nearly useless. Does the filter still track? What will the estimate look like?
3. **Tiny $\sigma_Q$** (~0.005): filter is told to trust its dynamics model. What goes wrong, and why?
4. **Enable the smoother.** Where does it help most — beginning, middle, end?
5. **Adding acceleration.** How does adding acceleration terms to $\Phi$ help the model in different scenarios?
6. **Changing the ground truth.** Play around with the `gen_truth` function at the top and observe how this affects your estimates. Which can be estimated with constant_velocity vs. constant_acceleration?

# What if we don't have values for $\Phi$?

On Tuesday, we also talked about state space models for measurements taken from bone marrow transfusion patients. Here we have measures of white blood cell count (WBC), platelets (PLT), and hematocrit (HCT) over 91 days. Early on in the dataset, there are measurements for each day, but as time goes on the measurements become more infrequent. 

We'd like to use a Kalman filter to estimate the states of each of these (WBC, PLT, HCT) while looking at interactions between them. We will fit the $\Phi$, $Q$, $R$, $\mu_0$, and $\sigma_0$ using maximum likelihood estimation (MLE).


In [ ]:
# Load the data

df=pd.read_csv('blood.csv')
df

# Modeling blood markers

We model three blood markers — log(WBC), log(PLT), and HCT — as a VAR(1) state-space model with missing observations. This matches the Shumway & Stoffer formulation:

**State equation:**
$$x_t = \Phi x_{t-1} + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$

**Observation equation:**
$$y_t = A_t x_t + v_t, \quad v_t \sim \mathcal{N}(0, R)$$

where $A_t = I_3$ when a blood sample is taken on day $t$ and $A_t = 0$ when no sample is available. The state $x_t = [\text{WBC}_t, \text{PLT}_t, \text{HCT}_t]^\top$ represents the underlying (unobserved) true marker levels.

**What we estimate via MLE:**
* $\Phi$: $(3 \times 3)$ transition matrix (9 parameters). This represents how markers evolve and interact from one day to the next.
* $Q$: $(3 \times 3)$ symmetric state noise covariance (6 parameters). This represents unmodeled biological variability
* $R$: $(3 \times 3)$ symmetric observation noise covariance (6 parameters). This represents measurement error

**What the Kalman filter/smoother computes:**
* Filtered estimates $x_t^t = E[x_t \mid y_{1:t}]$ for real-time tracking
* Smoothed estimates $x_t^n = E[x_t \mid y_{1:n}]$ for the best estimate using all data, including future observations
* Forecasts $x_{n+h}^n$ for prediction beyond the observation window [e.g., platelet count at day 100]

In [ ]:
from statsmodels.tsa.statespace.mlemodel import MLEModel

In [ ]:
# Look at the raw data and see what is missing
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
labels = ['log(WBC)', 'log(PLT)', 'HCT']
cols = ['WBC', 'PLT', 'HCT']

for ax, col, label in zip(axes, cols, labels):
    mask = df[col].notna()
    ax.plot(df.loc[mask, 'day'], df.loc[mask, col], 'o',
            color='#D85A30', ms=4, alpha=0.7)
    ax.set_ylabel(label)
    # Shade missing regions
    missing = df[col].isna().values
    for i in range(len(missing)):
        if missing[i]:
            ax.axvspan(df['day'].iloc[i] - 0.5, df['day'].iloc[i] + 0.5,
                       color='#888780', alpha=0.06)

axes[-1].set_xlabel('Day post-transplant')
axes[0].set_title('Raw observations (gray = missing)')
plt.tight_layout()
plt.show()

## Define the state-space model

We subclass `statsmodels.tsa.statespace.MLEModel` to define our model. We will relate our lecture notation to the variable names defined in `statsmodels`:

| Our notation | statsmodels name | Set in code |
|:---:|:---:|:---|
| $\Phi$ | `transition` | Estimated (9 params) |
| $A_t$ | `design` | Fixed as $I_3$; NaN rows handled automatically |
| $Q$ | `state_cov` | Estimated via Cholesky (6 params) |
| $R$ | `obs_cov` | Estimated via Cholesky (6 params) |
| $I$ | `selection` | Fixed as $I_3$ |

$Q$ and $R$ are parameterized as $L L^\top$ where $L$ is lower triangular. This guarantees positive semi-definiteness regardless of what the optimizer tries.

In [ ]:
class BloodSSM(MLEModel):
    """
    x_t = Phi @ x_{t-1} + w_t,   w_t ~ N(0, Q)
    y_t = A_t @ x_t + v_t,        v_t ~ N(0, R)

    A_t = I when observed; statsmodels zeros out rows for NaN
    automatically. Parameters: Phi (9), Q (6 Cholesky), R (6 Cholesky).
    """

    def __init__(self, endog):
        super().__init__(endog, k_states=3, k_posdef=3,
                         initialization='diffuse')
        self['design'] = np.eye(3)
        self['selection'] = np.eye(3)

    @staticmethod
    def _params_to_lower(params, k=3):
        L = np.zeros((k, k))
        L[np.tril_indices(k)] = np.real(params[:k * (k + 1) // 2])
        return L

    @property
    def param_names(self):
        phi = [f'phi.{i+1}{j+1}' for i in range(3) for j in range(3)]
        cQ = [f'chol_Q.{i+1}{j+1}' for i in range(3) for j in range(i + 1)]
        cR = [f'chol_R.{i+1}{j+1}' for i in range(3) for j in range(i + 1)]
        return phi + cQ + cR

    @property
    def start_params(self):
        # Initial Phi: near-identity (persistent markers)
        Phi_init = np.array([[0.9, 0.0, 0.0],
                             [0.0, 0.9, 0.0],
                             [0.0, 0.0, 0.9]])

        # Initial Cholesky factors for Q and R
        chol_Q_init = np.array([[0.1, 0.0, 0.0],
                                [0.0, 0.1, 0.0],
                                [0.0, 0.0, 1.0]])

        chol_R_init = np.array([[0.1, 0.0, 0.0],
                                [0.0, 0.1, 0.0],
                                [0.0, 0.0, 1.0]])

        # Flatten for optimizer (internal detail)
        return np.concatenate([
            Phi_init.ravel(),
            chol_Q_init[np.tril_indices(3)],
            chol_R_init[np.tril_indices(3)],
        ])

    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        Phi_full = np.real(params[:9]).reshape(3,3)
        #self['transition'] = np.diag(np.diag(Phi_full)) # uncomment for diagonal only matrix, no interactions
        self['transition'] = np.real(params[:9]).reshape(3, 3)
        L_Q = self._params_to_lower(params[9:15])
        L_R = self._params_to_lower(params[15:21])
        self['state_cov'] = L_Q @ L_Q.T
        self['obs_cov'] = L_R @ L_R.T

# Fit the model

In [ ]:
endog = df[['WBC', 'PLT', 'HCT']].values.astype(float)

mod = BloodSSM(endog)
res = mod.fit(disp=False, maxiter=2000, method='powell', cov_type='robust')
print(res.summary())

Notice here that you may get errors related to the covariance matrix being singular or near-singular. The Ljung-Box test tests whether our one-step-ahead prediction errors are serially correlated - if these p-values are less than 0.05, then we do have correlations left in our model that we're missing out on. Here it seems like that's not the case, so that's good news. 

On the other hand, there are a number of other assumptions that we aren't meeting. The standard errors here are all close to zero, which means the confidence intervals are likely not trustworthy. Although there are other diagnostics here that are maybe concerning, this is often the case with real biological data. One thing we could try is to force the dynamics to be simpler (i.e. only allow for diagonal terms in $\Phi$), and see if this helps. 

## Extract and interpret estimated matrices

In this scenario, we are estimating $\Phi$, $Q$, $R$, so we will print those here and talk about how to interpret them.

In [ ]:
# Helper to extract Q or R from Cholesky parameters
def extract_chol_matrix(params, start_idx):
    L = np.zeros((3, 3))
    idx = start_idx
    for i in range(3):
        for j in range(i + 1):
            L[i, j] = np.real(params[idx]); idx += 1
    return L @ L.T, idx

params = res.params
Phi_hat = np.real(params[0:9]).reshape(3, 3)
Q_hat, next_idx = extract_chol_matrix(params, 9)
R_hat, _ = extract_chol_matrix(params, next_idx)

marker_names = ['WBC', 'PLT', 'HCT']

print("Estimated Phi:")
print(pd.DataFrame(Phi_hat, index=marker_names, columns=marker_names).round(3))
print(f"\nEstimated Q:")
print(pd.DataFrame(Q_hat, index=marker_names, columns=marker_names).round(4))
print(f"\nEstimated R:")
print(pd.DataFrame(R_hat, index=marker_names, columns=marker_names).round(4))

### Interpreting $\hat{\Phi}$

Each row of $\Phi$ is a regression: today's marker value as a function of yesterday's three markers.

**Diagonal entries (persistence):**
- $\phi_{11} \approx 0.95$: log(WBC) is highly persistent — today's value is close to yesterday's
- $\phi_{22} \approx 0.91$: log(PLT) is slightly less persistent
- $\phi_{33} \approx 0.89$: HCT is the least persistent of the three

All diagonal entries are close to 1 but strictly less, consistent with mean-reverting biological processes. A value of exactly 1 would imply a random walk (no mean reversion).

**Off-diagonal entries (cross-coupling):**
- $\phi_{21} \approx 0.07$: yesterday's WBC slightly predicts today's PLT (consistent with WBC recovery preceding platelet recovery during engraftment)
- $\phi_{31} \approx -0.79$ and $\phi_{32} \approx 1.21$: HCT is strongly coupled to both WBC and PLT — its dynamics are driven more by the other markers than by its own history

## Plot the Kalman filter and smoother estimates

The Kalman smoother gives us $x_t^n = E[x_t \mid y_{1:n}]$, which is the best estimate of each marker at every time point using the entire dataset. This fills in the missing values optimally. We can also compare this to the Kalman filter, which uses only the forward step and cannot integrate data from the future.

In [ ]:
from ipywidgets import Checkbox, Layout, interactive_output, VBox, HBox
from IPython.display import display

def contiguous_regions(mask):
    """Find contiguous True regions in a boolean array."""
    regions = []
    start = None
    for i, val in enumerate(mask):
        if val and start is None:
            start = i
        elif not val and start is not None:
            regions.append((start, i - 1))
            start = None
    if start is not None:
        regions.append((start, len(mask) - 1))
    return regions
    
# Extract filtered states and standard errors
filtered_state = res.filter_results.filtered_state.T
filtered_cov = res.filter_results.filtered_state_cov
filtered_se = np.sqrt(
    np.array([filtered_cov[i, i, :] for i in range(3)]).T
)

# Extract smoothed states and standard errors
smoothed_state = res.smoother_results.smoothed_state.T
smoothed_cov = res.smoother_results.smoothed_state_cov
smoothed_se = np.sqrt(
    np.array([smoothed_cov[i, i, :] for i in range(3)]).T
)

# Forecast to day 100
fc = res.get_forecast(steps=9)
fc_mean = fc.predicted_mean
fc_ci = fc.conf_int()

# Plot function
labels = ['log(WBC)', 'log(PLT)', 'HCT']
colors_smooth = ['#378ADD', '#1D9E75', '#534AB7']
colors_filter = ['#85B7EB', '#5DCAA5', '#AFA9EC']
days = df['day'].values
fc_days = np.arange(days[-1] + 1, days[-1] + 10)

def plot_blood(show_observed=True, show_filtered=True,
               show_smoothed=True, show_forecast=True,
               show_ci=True):
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    for i, (ax, label) in enumerate(zip(axes, labels)):
        obs = endog[:, i]
        observed_mask = ~np.isnan(obs)
        missing_mask = np.isnan(obs)

        # Shade missing regions
        for start, end in contiguous_regions(missing_mask):
            ax.axvspan(days[start] - 0.5,
                       days[min(end, len(days) - 1)] + 0.5,
                       color='#888780', alpha=0.06)

        if show_observed:
            ax.plot(days[observed_mask], obs[observed_mask], 'o',
                    color='#D85A30', ms=4, alpha=0.7,
                    label='Observed', zorder=3)

        if show_filtered:
            ax.plot(days, filtered_state[:, i], '-',
                    color=colors_filter[i], lw=1.2,
                    label='Filtered', alpha=0.8)
            if show_ci:
                ax.fill_between(days,
                    filtered_state[:, i] - 1.96 * filtered_se[:, i],
                    filtered_state[:, i] + 1.96 * filtered_se[:, i],
                    color=colors_filter[i], alpha=0.08)

        if show_smoothed:
            ax.plot(days, smoothed_state[:, i], '-',
                    color=colors_smooth[i], lw=1.5, label='Smoothed')
            if show_ci:
                ax.fill_between(days,
                    smoothed_state[:, i] - 1.96 * smoothed_se[:, i],
                    smoothed_state[:, i] + 1.96 * smoothed_se[:, i],
                    color=colors_smooth[i], alpha=0.12)

        if show_forecast:
            ax.plot(fc_days, fc_mean[:, i], '--',
                    color=colors_smooth[i], lw=1.2, label='Forecast')
            if show_ci:
                ax.fill_between(fc_days, fc_ci[:, i], fc_ci[:, i + 3],
                                color=colors_smooth[i], alpha=0.08)

        ax.set_ylabel(label)
        ax.legend(frameon=False, fontsize=9, loc='upper left')

    axes[-1].set_xlabel('Day post-transplant')
    axes[0].set_title('Blood markers: Kalman filter and smoother')
    plt.tight_layout()
    plt.show()


# Widgets
obs_w = Checkbox(value=True, description='Observed')
filt_w = Checkbox(value=True, description='Filtered')
smooth_w = Checkbox(value=True, description='Smoothed')
fc_w = Checkbox(value=True, description='Forecast')
ci_w = Checkbox(value=True, description='95% CI bands')

out = interactive_output(plot_blood, {
    'show_observed': obs_w,
    'show_filtered': filt_w,
    'show_smoothed': smooth_w,
    'show_forecast': fc_w,
    'show_ci': ci_w,
})

display(VBox([HBox([obs_w, filt_w, smooth_w, fc_w, ci_w]), out]))

### What do we see in these plots?

1. **Confidence intervals widen in missing-data regions** (gray bands) because no update step occurs. We can only use the dynamics, and $P_t$ grows via the $+Q$ term at each predict step. When the next observation arrives, $P_t$ snaps back down.

2. **WBC and PLT** are smoothly estimated with tight CIs even when there is missing data. Their small $Q$ entries mean the model treats them as slowly varying.

3. **HCT** has much wider CIs, consistent with its large $Q_{33}$ and $R_{33}$. Biologically, hematocrit is more variable and harder to measure precisely.

4. **Forecasts** (dashed lines past day 91) converge toward the long-run mean with widening CIs. The platelet forecast at day 100 is a clinically relevant quantity that is correlated with long term survival. In particular, research has shown a good prognosis for a total platelet count > 50 (log platelet count > 3.91) ([Bolwell et al. 2004](https://www.nature.com/articles/1704330).)

# Platelet forecast at day 100

Let's actually do the forecast up to day 100. The last day of measurements is day 91, so we need to do 9 more steps (0-8):

In [ ]:
day_100_idx = 8  # 9 steps ahead, 0-indexed
plt_forecast = fc_mean[day_100_idx, 1]
plt_ci_lo = fc_ci[day_100_idx, 1]
plt_ci_hi = fc_ci[day_100_idx, 4]

print(f"Day 100 log(PLT) forecast: {plt_forecast:.3f}")
print(f"95% CI: [{plt_ci_lo:.3f}, {plt_ci_hi:.3f}]")

# We can also convert the log(PLT) back to actual platelet count, 
# which we reference from the paper (Bolwell et al):
print(f"\nBack-transformed (PLT count):")
print(f"  Point estimate: {np.exp(plt_forecast):.0f}")
print(f"  95% CI: [{np.exp(plt_ci_lo):.0f}, {np.exp(plt_ci_hi):.0f}]")

## How do we connect these to the 2D tracking demo?

The blood marker model and the 2D tracking model are structurally identical. Both are linear Gaussian state-space models estimated by the Kalman filter/smoother. The key differences can be summarized in the table below:

| | Tracking demo | Blood markers |
|:---|:---|:---|
| $\Phi$ | Fixed by physics (kinematics) | Estimated from data (VAR(1)) |
| $A_t$ | Fixed as $I$ (always observe) | Time-varying: $I$ or $0$ (missing data) |
| $Q, R$ | Set by slider | Estimated via MLE |
| State meaning | Position + velocity | WBC + PLT + HCT |
| Estimation | Filter/smoother only | MLE for parameters, then filter/smoother for states |

In all of these cases, we use the same filter equations, and we also use the same prediction + update with Kalman gain. However, in the case of the first model, we have a physical model and its dynamics we can rely on, while in the second, we have to estimate these parameters.